In [1]:
#importing packages and vars

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

#load environment variables
env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [2]:
spark = ((SparkSession
            .builder
            .appName('dimcountry')
            .config('spark.jar', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
            
        ))

your 131072x1 screen size is bogus. expect trouble
24/04/25 15:37:35 WARN Utils: Your hostname, DOUGPC resolves to a loopback address: 127.0.1.1; using 172.30.204.147 instead (on interface eth0)
24/04/25 15:37:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/25 15:37:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/04/25 15:37:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
tmp_countryframe = (( spark
                        .read
                        .format('jdbc')
                        .option('url', f'{mssqlsource}')
                        .option('driver', f'{mssqldriver}')
                        .option('user', f'{mssqluser}')
                        .option('password', f'{mssqlpwd}')
                        .option('query', 
                                
                                """ 
                                select

                                    ac.countryid,
                                    ac.continent,
                                    ac.region,
                                    ac.subregion,
                                    ac.formalname,
                                    ac.countryname,
                                    ac.isonumericcode,
                                    ac.isoalpha3code,
                                    ac.latestrecordedpopulation
                                                                  
                                from application.countries as ac
                                
                                """
                            )
                     
                )).load()

In [4]:
#renaming columns

countryframe = (( tmp_countryframe
                    .withColumnRenamed('isonumericcode', 'isostandardid')
                    .withColumnRenamed('isoalpha3code', 'countrycode')
                    .withColumnRenamed('region', 'countryregion')
                 
                ))

In [5]:
(( 
  countryframe
    .write
    .format('jdbc')
    .option('url', f'{mssql_dwdest}')
    .option('driver', f'{mssqldriver}')
    .option('user', f'{mssqluser}')
    .option('password', f'{mssqlpwd}')
    .option('dbtable', 'dimension.country')
    .save(mode= 'overwrite')
))